# Comparación de modelos: TDA vs. sin TDA

Extensión de la Sesión 5. La idea es **reutilizar tu lógica de TDA tal cual** y
cambiar únicamente el estimador. Para eso generalizamos `entrenar_y_evaluar`
para que reciba un `modelo` (un estimador de scikit-learn sin entrenar) en lugar
de tener el `RandomForestRegressor` fijo por dentro.

Luego corremos un **barrido de modelos × {sin TDA, con TDA}** sobre un par
entrenamiento→prueba y comparamos el % de mejora del MAE para cada modelo.

> Requisito: tener ya ejecutadas las celdas originales de **Configuración inicial**,
> **Carga de datos** y la definición de `extraer_datos_por_fecha` /
> `periodos_entrenamiento`. Este notebook reemplaza la función `entrenar_y_evaluar`
> por una versión parametrizada (compatible: si no pasas `modelo`, usa Random Forest).

In [ ]:
# ============================================================
# FUNCIÓN BASE GENERALIZADA: ahora recibe `modelo`
# (misma lógica TDA y predicción recursiva que la original)
# ============================================================
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor

def entrenar_y_evaluar(signal_entreno, signal_prueba,
                       window_size=24, embedding_dim=6, time_delay=3,
                       use_tda=True, modelo=None):
    \"\"\"Igual que la original, pero el estimador es intercambiable.

    - modelo: estimador sklearn SIN entrenar. Si es None -> Random Forest (default).
              Se clona internamente para no reutilizar el mismo objeto entre llamadas.
    \"\"\"
    if modelo is None:
        modelo = RandomForestRegressor(n_estimators=200, max_depth=10,
                                       min_samples_split=5, random_state=42, n_jobs=-1)

    # --- Ventanas deslizantes ---
    SW = SlidingWindow(size=window_size, stride=1)
    X_signal = signal_entreno.reshape(-1, 1)
    X_windows, y_train = SW.fit_transform_resample(X_signal, signal_entreno)

    if use_tda:
        STE = SingleTakensEmbedding(parameters_type="fixed",
                                    dimension=embedding_dim, time_delay=time_delay)
        VR = VietorisRipsPersistence(homology_dimensions=[1], n_jobs=-1, collapse_edges=True)

        topo_features = []
        for i in range(X_windows.shape[0]):
            window_1d = X_windows[i].flatten()
            try:
                embedding = STE.fit_transform(window_1d)
                if len(embedding) >= 5:
                    diagrams = VR.fit_transform([embedding])[0]
                    mask = diagrams[:, 2] == 1
                    if np.any(mask):
                        p = diagrams[mask, 1] - diagrams[mask, 0]
                        topo_features.append([np.max(p), np.mean(p), np.std(p), len(p)])
                    else:
                        topo_features.append([0, 0, 0, 0])
                else:
                    topo_features.append([0, 0, 0, 0])
            except Exception:
                topo_features.append([0, 0, 0, 0])
        topo_features = np.array(topo_features)

        scaler_raw = StandardScaler()
        scaler_topo = StandardScaler()
        X_raw_norm = scaler_raw.fit_transform(X_windows.reshape(-1, window_size))
        X_topo_norm = scaler_topo.fit_transform(topo_features)
        X_train = np.hstack([X_raw_norm, X_topo_norm])
    else:
        scaler_raw = StandardScaler()
        X_train = scaler_raw.fit_transform(X_windows.reshape(-1, window_size))

    # --- Entrenar el modelo intercambiable ---
    modelo = clone(modelo)
    modelo.fit(X_train, y_train)

    # --- Predicción recursiva (idéntica, usa `modelo`) ---
    preds = []
    current_window = signal_entreno[-window_size:].copy()
    for i in range(len(signal_prueba)):
        if use_tda:
            try:
                STE_p = SingleTakensEmbedding(parameters_type="fixed",
                                              dimension=embedding_dim, time_delay=time_delay)
                VR_p = VietorisRipsPersistence(homology_dimensions=[1], n_jobs=-1, collapse_edges=True)
                embedding = STE_p.fit_transform(current_window)
                if len(embedding) >= 5:
                    diagrams = VR_p.fit_transform([embedding])[0]
                    mask = diagrams[:, 2] == 1
                    if np.any(mask):
                        p = diagrams[mask, 1] - diagrams[mask, 0]
                        topo_feat = np.array([np.max(p), np.mean(p), np.std(p), len(p)]).reshape(1, -1)
                    else:
                        topo_feat = np.array([[0, 0, 0, 0]])
                else:
                    topo_feat = np.array([[0, 0, 0, 0]])
                x_pred = np.hstack([scaler_raw.transform(current_window.reshape(1, -1)),
                                    scaler_topo.transform(topo_feat)])
            except Exception:
                x_pred = scaler_raw.transform(current_window.reshape(1, -1))
        else:
            x_pred = scaler_raw.transform(current_window.reshape(1, -1))

        pred = modelo.predict(x_pred)[0]
        preds.append(pred)
        if i < len(signal_prueba) - 1:
            current_window = np.roll(current_window, -1)
            current_window[-1] = signal_prueba[i]

    mae = mean_absolute_error(signal_prueba, preds)
    rmse = np.sqrt(mean_squared_error(signal_prueba, preds))
    r2 = r2_score(signal_prueba, preds)
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'preds': preds}

print("✅ entrenar_y_evaluar() ahora acepta `modelo`")

## Catálogo de modelos a comparar

Por qué estos (todos usan las features ya escaladas, así que conviven con TDA):

- **Ridge / Lasso / ElasticNet** — lineales regularizados. Baseline rápido y estable
  con pocas muestras; útiles para ver si las 4 features topológicas aportan señal
  *lineal*.
- **SVR (RBF)** — fuerte en datasets pequeños y no lineales; muy sensible a la escala
  (ya la tienes). Suele ser donde el TDA luce más.
- **KNN** — predice por vecinos en el espacio de features; como el TDA cambia la
  geometría del espacio, es interesante para ver el efecto del TDA aislado.
- **ExtraTrees / GradientBoosting / HistGradientBoosting** — ensembles de árboles,
  familia del Random Forest; comparan contra tu baseline actual.
- **MLP** — red neuronal pequeña; puede combinar raw + topo, pero ojo con overfitting
  si hay pocas ventanas.
- **XGBoost / LightGBM** — opcionales (solo si están instalados).

⚠️ Con periodos cortos (p. ej. 2020–2022 = 36 meses y `window=24` deja ~12 ventanas)
los modelos flexibles (MLP, boosting) pueden sobreajustar. Lineales, SVR y KNN suelen
ser más seguros ahí.

In [ ]:
# ============================================================
# CATÁLOGO DE MODELOS
# ============================================================
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor, HistGradientBoostingRegressor)
from sklearn.neural_network import MLPRegressor

modelos = {
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=10,
                                            min_samples_split=5, random_state=42, n_jobs=-1),
    "Extra Trees":   ExtraTreesRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "Grad. Boosting": GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                                learning_rate=0.05, random_state=42),
    "Hist GB":       HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05,
                                                   random_state=42),
    "Ridge":         Ridge(alpha=1.0),
    "Lasso":         Lasso(alpha=0.1, max_iter=10000),
    "ElasticNet":    ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000),
    "SVR (RBF)":     SVR(kernel="rbf", C=10, gamma="scale"),
    "KNN":           KNeighborsRegressor(n_neighbors=5, weights="distance"),
    "MLP":           MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=2000,
                                  random_state=42, early_stopping=True),
}

# Opcionales: se añaden solo si la librería está disponible
try:
    from xgboost import XGBRegressor
    modelos["XGBoost"] = XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05,
                                      random_state=42, n_jobs=-1, verbosity=0)
except Exception:
    print("ℹ️ XGBoost no instalado, se omite (pip install xgboost)")

try:
    from lightgbm import LGBMRegressor
    modelos["LightGBM"] = LGBMRegressor(n_estimators=300, max_depth=3, learning_rate=0.05,
                                        random_state=42, n_jobs=-1, verbose=-1)
except Exception:
    print("ℹ️ LightGBM no instalado, se omite (pip install lightgbm)")

print(f"✅ {len(modelos)} modelos en el catálogo:", ", ".join(modelos))

## Barrido: cada modelo, con y sin TDA

Fijamos un par entrenamiento→prueba y parámetros de TDA fijos
(`embedding_dim=3`, `time_delay=1`) para que la *única* diferencia entre las dos
columnas de cada modelo sea la presencia de las features topológicas.

In [ ]:
# ============================================================
# BARRIDO DE MODELOS  (un periodo)
# ============================================================
import pandas as pd

# Elige el par a evaluar (igual que en tus actividades)
signal_entreno, _ = extraer_datos_por_fecha('2013-01-01', '2019-12-31')   # entreno
signal_prueba,  _ = extraer_datos_por_fecha('2020-01-01', '2022-12-31')   # prueba

WINDOW, DIM, TAU = 24, 3, 1   # mismos parámetros para todos, para una comparación justa

filas = []
for nombre, modelo in modelos.items():
    base = entrenar_y_evaluar(signal_entreno, signal_prueba,
                              window_size=WINDOW, use_tda=False, modelo=modelo)
    tda  = entrenar_y_evaluar(signal_entreno, signal_prueba,
                              window_size=WINDOW, embedding_dim=DIM, time_delay=TAU,
                              use_tda=True, modelo=modelo)
    mejora = (base['mae'] - tda['mae']) / base['mae'] * 100
    filas.append({
        'Modelo': nombre,
        'MAE_sinTDA': base['mae'], 'MAE_conTDA': tda['mae'],
        'R2_sinTDA': base['r2'],  'R2_conTDA': tda['r2'],
        'Mejora_%': mejora,
    })
    print(f"{nombre:16s} | MAE {base['mae']:7.2f} -> {tda['mae']:7.2f} | mejora {mejora:+6.2f}%")

df_modelos = pd.DataFrame(filas).sort_values('MAE_conTDA').reset_index(drop=True)
print("\n=== TABLA ORDENADA POR MAE CON TDA ===")
print(df_modelos.to_string(index=False))

## Visualización comparativa

In [ ]:
# ============================================================
# GRÁFICOS: MAE (sin vs con TDA) y % de mejora por modelo
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

orden = df_modelos['Modelo'].tolist()
x = np.arange(len(orden)); w = 0.38

ax1 = axes[0]
ax1.bar(x - w/2, df_modelos['MAE_sinTDA'], w, label='Sin TDA', color='coral', edgecolor='black', alpha=0.85)
ax1.bar(x + w/2, df_modelos['MAE_conTDA'], w, label='Con TDA', color='steelblue', edgecolor='black', alpha=0.85)
ax1.set_xticks(x); ax1.set_xticklabels(orden, rotation=45, ha='right')
ax1.set_ylabel('MAE'); ax1.set_title('MAE por modelo: sin vs con TDA')
ax1.legend(); ax1.grid(True, alpha=0.3, axis='y')

ax2 = axes[1]
mejoras = df_modelos['Mejora_%']
colors = ['green' if m > 0 else 'red' for m in mejoras]
ax2.bar(x, mejoras, color=colors, edgecolor='black', alpha=0.85)
ax2.axhline(0, color='black', lw=1.2)
ax2.set_xticks(x); ax2.set_xticklabels(orden, rotation=45, ha='right')
ax2.set_ylabel('Mejora en MAE (%)'); ax2.set_title('¿Cuánto ayuda el TDA en cada modelo?')
ax2.grid(True, alpha=0.3, axis='y')
for xi, m in zip(x, mejoras):
    ax2.text(xi, m + (0.4 if m > 0 else -0.9), f'{m:+.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Comparación de modelos — Entreno 2013-2019 → Prueba 2020-2022', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## (Opcional) El mismo barrido en todos los periodos

Reaprovecha `pares_periodos` de tu Actividad 1 para ver qué modelo es más robusto
y en qué regímenes el TDA aporta. Devuelve la mejora (%) por modelo × periodo.

In [ ]:
# ============================================================
# BARRIDO MODELOS × PERIODOS  (matriz de mejora % del TDA)
# ============================================================
matriz = {}
for nombre, modelo in modelos.items():
    fila = {}
    for ne, npr, desc in pares_periodos:
        se, _ = extraer_datos_por_fecha(*periodos_entrenamiento[ne])
        sp, _ = extraer_datos_por_fecha(*periodos_entrenamiento[npr])
        try:
            base = entrenar_y_evaluar(se, sp, window_size=WINDOW, use_tda=False, modelo=modelo)
            tda  = entrenar_y_evaluar(se, sp, window_size=WINDOW, embedding_dim=DIM,
                                      time_delay=TAU, use_tda=True, modelo=modelo)
            fila[desc] = (base['mae'] - tda['mae']) / base['mae'] * 100
        except Exception as e:
            fila[desc] = np.nan
    matriz[nombre] = fila
    print(f"{nombre:16s}", {k: round(v, 1) for k, v in fila.items()})

df_matriz = pd.DataFrame(matriz).T  # filas=modelos, columnas=periodos
print("\n=== MEJORA % DEL TDA (modelo × periodo) ===")
print(df_matriz.round(2).to_string())

# Heatmap
fig, ax = plt.subplots(figsize=(12, 0.6 * len(df_matriz) + 2))
im = ax.imshow(df_matriz.values, cmap='RdYlGn', aspect='auto',
               vmin=-np.nanmax(np.abs(df_matriz.values)), vmax=np.nanmax(np.abs(df_matriz.values)))
ax.set_xticks(range(df_matriz.shape[1])); ax.set_xticklabels(df_matriz.columns, rotation=30, ha='right')
ax.set_yticks(range(df_matriz.shape[0])); ax.set_yticklabels(df_matriz.index)
for i in range(df_matriz.shape[0]):
    for j in range(df_matriz.shape[1]):
        v = df_matriz.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:+.0f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, label='Mejora MAE (%)')
ax.set_title('Mejora del TDA por modelo y periodo', fontweight='bold')
plt.tight_layout(); plt.show()

## Notas e interpretación

- **Comparación justa:** dentro de cada modelo, lo único que cambia entre las dos
  columnas son las 4 features topológicas; el resto (ventana, escalado,
  predicción recursiva) es idéntico. El % de mejora es atribuible al TDA.
- **Que el TDA ayude depende del modelo:** suele notarse más en SVR/KNN/lineales
  (que dependen mucho de la geometría de las features) que en ensembles de árboles,
  que ya capturan interacciones por su cuenta.
- **Muestras pocas → cuidado con overfitting** en MLP/boosting; mira si `R2_conTDA`
  cae respecto a `R2_sinTDA`.
- **Baseline clásico (opcional):** un modelo puro de series como SARIMA/ETS no encaja
  en el switch TDA/no-TDA (no usa estas features), pero sirve como referencia externa
  de "¿el enfoque ventana+ML+TDA supera a un modelo de series estándar?". Si te
  interesa, lo agregamos aparte con `statsmodels`.
- Puedes afinar hiperparámetros por modelo con `GridSearchCV` usando
  `TimeSeriesSplit` para no mezclar futuro con pasado.